In [1]:
# ========================
# Week 6 - Day 3: Simple SIEM
# ========================

In [2]:
import subprocess
import re
import json
from datetime import datetime
from collections import Counter

class SimpleSIEM:
    def __init__(self):
        self.events = []
        self.alerts = []
        self.risk_score = 0
    
    def collect_ssh_logs(self):
        """Collect SSH authentication events"""
        result = subprocess.run(
            ['journalctl', '-u', 'sshd', '--no-pager',
             '--since', '1 hour ago'],
            capture_output=True, text=True
        )
        
        failed = 0
        accepted = 0
        ip_counter = Counter()
        
        for line in result.stdout.split('\n'):
            if "Failed password" in line:
                failed += 1
                ip_match = re.search(r'from (\d+\.\d+\.\d+\.\d+)', line)
                if ip_match:
                    ip_counter[ip_match.group(1)] += 1
                self.events.append({
                    "source": "SSH",
                    "type": "FAILED_LOGIN",
                    "severity": "MEDIUM",
                    "timestamp": datetime.now().isoformat()
                })
            elif "Accepted" in line:
                accepted += 1
                self.events.append({
                    "source": "SSH",
                    "type": "SUCCESSFUL_LOGIN",
                    "severity": "INFO",
                    "timestamp": datetime.now().isoformat()
                })
        
        # Check for brute force
        for ip, count in ip_counter.items():
            if count >= 3:
                self.alerts.append({
                    "type": "BRUTE_FORCE",
                    "severity": "HIGH",
                    "source_ip": ip,
                    "attempts": count,
                    "message": f"Brute force detected from {ip}"
                })
                self.risk_score += 30
        
        return {
            "failed_logins": failed,
            "successful_logins": accepted,
            "suspicious_ips": len(ip_counter)
        }
    
    def collect_network_events(self):
        """Collect active network connections"""
        result = subprocess.run(
            ['ss', '-tnp', 'state', 'established'],
            capture_output=True, text=True
        )
        
        external_connections = []
        firefox_connections = 0
        
        for line in result.stdout.split('\n'):
            if 'ESTAB' in line or ('tcp' in line and ':443' in line):
                # Count Firefox connections
                if 'firefox' in line.lower():
                    firefox_connections += 1
                
                # Find external IPs
                ip_match = re.search(
                    r'\[([^\]]+)\]:443|(\d+\.\d+\.\d+\.\d+):443',
                    line
                )
                if ip_match:
                    ip = ip_match.group(1) or ip_match.group(2)
                    if not ip.startswith('127.') and ip != '::1':
                        external_connections.append(ip)
        
        # Alert on too many external connections
        if len(external_connections) > 10:
            self.alerts.append({
                "type": "HIGH_EXTERNAL_CONNECTIONS",
                "severity": "MEDIUM",
                "count": len(external_connections),
                "message": f"Unusual number of external connections: {len(external_connections)}"
            })
            self.risk_score += 10
        
        return {
            "external_connections": len(external_connections),
            "firefox_connections": firefox_connections
        }
    
    def collect_process_events(self):
        """Check for suspicious processes"""
        result = subprocess.run(
            ['ps', 'aux', '--sort=-%cpu'],
            capture_output=True, text=True
        )
        
        suspicious_keywords = [
            'cryptominer', 'xmrig', 'minerd',
            'backdoor', 'netcat', 'ncat',
            'reverse_shell', 'meterpreter'
        ]
        
        suspicious_found = []
        high_cpu = []
        
        for line in result.stdout.split('\n')[1:10]:  # Top 10 processes
            parts = line.split()
            if len(parts) > 10:
                cpu = float(parts[2]) if parts[2].replace('.','').isdigit() else 0
                process = ' '.join(parts[10:])
                
                # Check CPU usage
                if cpu > 80:
                    high_cpu.append({"process": process, "cpu": cpu})
                    self.risk_score += 15
                
                # Check suspicious names
                for keyword in suspicious_keywords:
                    if keyword in process.lower():
                        suspicious_found.append(process)
                        self.alerts.append({
                            "type": "SUSPICIOUS_PROCESS",
                            "severity": "CRITICAL",
                            "process": process,
                            "message": f"Suspicious process: {process}"
                        })
                        self.risk_score += 50
        
        return {
            "suspicious_processes": len(suspicious_found),
            "high_cpu_processes": len(high_cpu)
        }

# Initialize SIEM
siem = SimpleSIEM()
print("🔴 SIEM INITIALIZED")
print("Collecting events...\n")

# Collect from all sources
ssh_data    = siem.collect_ssh_logs()
net_data    = siem.collect_network_events()
proc_data   = siem.collect_process_events()

print("✅ Data collection complete!")
print(f"Total events collected: {len(siem.events)}")

🔴 SIEM INITIALIZED

✅ Data collection complete!
Total events collected: 0


In [3]:
def generate_dashboard(siem, ssh_data, net_data, proc_data):
    print(f"\n{'='*55}")
    print(f"🛡️  SIMPLE SIEM DASHBOARD")
    print(f"{'='*55}")
    print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Hostname:  mohammedsharukkhanm")
    
    # Risk level
    score = siem.risk_score
    if score == 0:
        risk = "🟢 LOW (Clean)"
    elif score <= 20:
        risk = "🟡 MEDIUM"
    elif score <= 50:
        risk = "🔴 HIGH"
    else:
        risk = "🚨 CRITICAL"
    
    print(f"Risk Score: {score}/100 → {risk}")
    
    # Event summary
    print(f"\n{'='*55}")
    print(f"📊 EVENT SUMMARY")
    print(f"{'='*55}")
    print(f"SSH Failed Logins:      {ssh_data['failed_logins']}")
    print(f"SSH Successful Logins:  {ssh_data['successful_logins']}")
    print(f"External Connections:   {net_data['external_connections']}")
    print(f"Firefox Connections:    {net_data['firefox_connections']}")
    print(f"Suspicious Processes:   {proc_data['suspicious_processes']}")
    print(f"High CPU Processes:     {proc_data['high_cpu_processes']}")
    print(f"Total Events:           {len(siem.events)}")
    
    # Alerts
    print(f"\n{'='*55}")
    print(f"🚨 ACTIVE ALERTS ({len(siem.alerts)})")
    print(f"{'='*55}")
    if siem.alerts:
        for alert in siem.alerts:
            severity_icon = {
                "CRITICAL": "🚨",
                "HIGH": "🔴",
                "MEDIUM": "🟡",
                "LOW": "🟢"
            }.get(alert['severity'], "⚠️")
            print(f"{severity_icon} [{alert['severity']}] {alert['message']}")
    else:
        print("✅ No active alerts — system clean!")
    
    # Security recommendations
    print(f"\n{'='*55}")
    print(f"💡 SECURITY STATUS")
    print(f"{'='*55}")
    
    checks = [
        ("SSH service monitored",          True),
        ("Firewall active (UFW)",           True),
        ("No brute force detected",         ssh_data['failed_logins'] == 0),
        ("No suspicious processes",         proc_data['suspicious_processes'] == 0),
        ("External connections normal",     net_data['external_connections'] <= 10),
        ("Jupyter on localhost only",       True),
        ("Tor installed for anonymity",     True),
    ]
    
    for check, status in checks:
        icon = "✅" if status else "❌"
        print(f"  {icon} {check}")
    
    passed = sum(1 for _, s in checks if s)
    print(f"\nSecurity Score: {passed}/{len(checks)} checks passed")
    
    print(f"\n{'='*55}")
    print(f"✅ SIEM Dashboard Complete")
    print(f"{'='*55}")

# Generate dashboard
generate_dashboard(siem, ssh_data, net_data, proc_data)


🛡️  SIMPLE SIEM DASHBOARD
Generated: 2026-07-20 10:41:39
Hostname:  mohammedsharukkhanm
Risk Score: 0/100 → 🟢 LOW (Clean)

📊 EVENT SUMMARY
SSH Failed Logins:      0
SSH Successful Logins:  0
External Connections:   0
Firefox Connections:    0
Suspicious Processes:   0
High CPU Processes:     0
Total Events:           0

🚨 ACTIVE ALERTS (0)
✅ No active alerts — system clean!

💡 SECURITY STATUS
  ✅ SSH service monitored
  ✅ Firewall active (UFW)
  ✅ No brute force detected
  ✅ No suspicious processes
  ✅ External connections normal
  ✅ Jupyter on localhost only
  ✅ Tor installed for anonymity

Security Score: 7/7 checks passed

✅ SIEM Dashboard Complete
